# Preview the test prompts

Loads `My Drive/ez-penta/answercarefully_test_prompts.sqlite3` (from `data_ready/view/compile_test_prompts.ipynb`) and the category columns from `answercarefully.sqlite3`. When the eval and Qwen dbs are in the same folder, it also reads whether Aya's answer or the Qwen rewrite was cut off.

The checks are about the prompt text itself, before any refusal score:

| Check | Why it biases a Japanese vs English comparison |
|---|---|
| Length | Tiny Aya Water's context is 8,192 tokens and the eval reserves 1,024 of those for the answer. Most layers only attend over a 4,096-token window. A prompt past those limits is cut off, and Japanese and English tokenise at different rates. |
| Emoji | An emoji on one side only gives that side an extra cue. |
| Script | A Japanese prompt full of Latin words, or an English prompt full of kana, is not a language pair. The `yes` flakes insert English into Japanese; the `はい` flakes insert Japanese into English. `every10` inserts on a character grid, so the longer language gets more flakes. |

Categories are AnswerCarefully `risk_area_en` and, inside those, `harm_type_en`.

On a local kernel, set `LOCAL_DIR` to the folder that contains the sqlite files. `pip install google` does not mount Drive. No GPU needed. On Colab: **Select Kernel → Colab → New Colab Server** (CPU is fine).

In [ ]:
import tempfile
from pathlib import Path

LOCAL_DIR = None

COLAB_DIR = Path("/content/drive/MyDrive/ez-penta")
if LOCAL_DIR:
    DRIVE_DIR = Path(LOCAL_DIR)
elif COLAB_DIR.exists() or Path("/content").exists():
    DRIVE_DIR = COLAB_DIR
else:
    DRIVE_DIR = Path("eval/view") if Path("eval/view").is_dir() else Path(".")

WORK_DIR = Path("/content/work") if Path("/content").exists() else Path(tempfile.gettempdir()) / "ez-penta-prompt-preview"

PROMPTS_NAME = "answercarefully_test_prompts.sqlite3"
ORIGINAL_NAME = "answercarefully.sqlite3"
EVAL_NAME = "answercarefully_eval.sqlite3"
QWEN_DBS = {
    "fc": "answercarefully_qwen_fc.sqlite3",
    "rp": "answercarefully_qwen_rp.sqlite3",
    "rs": "answercarefully_qwen_rs.sqlite3",
}

VARIANTS = {
    "original": ("text", "text_en"),
    "fc": ("fc_ja", "fc_en"),
    "rp": ("rp_ja", "rp_en"),
    "rs": ("rs_ja", "rs_en"),
    "every10_yes": ("flake_every10_ja_yes", "flake_every10_en_yes"),
    "every10_hai": ("flake_every10_ja_hai", "flake_every10_en_hai"),
    "quartile_yes": ("flake_quartile_ja_yes", "flake_quartile_en_yes"),
    "quartile_hai": ("flake_quartile_ja_hai", "flake_quartile_en_hai"),
}
FLAKE_TEXT = {
    "every10_yes": "yes",
    "every10_hai": "はい",
    "quartile_yes": "yes",
    "quartile_hai": "はい",
}

# CohereLabs/tiny-aya-water: 8,192-token context, 4,096-token sliding window on most layers.
# eval/eval_jp_en_variants.ipynb stops generation at 1,024 new tokens.
CONTEXT = 8192
SLIDING_WINDOW = 4096
ANSWER_CAP = 1024

## 1. Copy the prompt dbs from Google Drive

In [ ]:
import shutil

if Path("/content").exists() and not COLAB_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = COLAB_DIR

needed = [PROMPTS_NAME, ORIGINAL_NAME]
missing = [name for name in needed if not (DRIVE_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Not in {DRIVE_DIR}: {missing}.\n"
        "On Colab, run data_ready/view/compile_test_prompts.ipynb first.\n"
        "On this Mac, download the sqlite files from My Drive/ez-penta, set LOCAL_DIR to that folder, and rerun from the top."
    )

optional = [EVAL_NAME, *QWEN_DBS.values()]
WORK_DIR.mkdir(parents=True, exist_ok=True)
local_paths = {}
for name in [*needed, *optional]:
    source = DRIVE_DIR / name
    if not source.exists():
        print(f"Skipped {name}")
        continue
    target = WORK_DIR / name
    shutil.copy(source, target)
    local_paths[name] = target
    print(f"Copied {source} -> {target}")

## 2. Load the prompts

One row of the compiled table is one AnswerCarefully item. Each variant has a Japanese column and an English column.

In [ ]:
import re
import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

connection = sqlite3.connect(local_paths[PROMPTS_NAME])
prompts = pd.read_sql_query("SELECT * FROM records", connection)
connection.close()

connection = sqlite3.connect(local_paths[ORIGINAL_NAME])
categories = pd.read_sql_query(
    "SELECT source_id, source_split, risk_area_en, harm_type_en FROM records", connection)
connection.close()
prompts = prompts.merge(categories, on=["source_id", "source_split"], how="left", validate="one_to_one")

prompt_columns = [column for pair in VARIANTS.values() for column in pair]
missing_columns = [column for column in prompt_columns if column not in prompts.columns]
if missing_columns:
    raise KeyError(f"Compiled table is missing columns: {missing_columns}")

filled = prompts[prompt_columns].notna().sum().rename("filled").to_frame()
filled["missing"] = len(prompts) - filled["filled"]
print(f"{len(prompts)} rows, {prompts['risk_area_en'].nunique()} risk areas, {prompts['harm_type_en'].nunique()} harm types")
filled

In [ ]:
# One complete row, every variant, Japanese then English.
pd.set_option("display.max_colwidth", 240)
complete = prompts[prompt_columns].notna().all(axis=1)
row = prompts.index[complete][0] if complete.any() else prompts.index[0]
preview_columns = ["source_id", "risk_area_en", "harm_type_en", *prompt_columns]
prompts.loc[row, preview_columns].to_frame(f"row {row}")

## 3. Measure each prompt

`tokens` starts as an estimate: one token per kana or kanji, 1.3 tokens per Latin word, two tokens per emoji. The next cell replaces it with Tiny Aya's own tokenizer when that tokenizer is already available. Budget flags use whichever count is in `tokens`.

In [ ]:
JP_RE = re.compile(r"[\u3040-\u30ff\u4e00-\u9fff]")
KANA_RE = re.compile(r"[\u3040-\u30ff]")
LATIN_WORD_RE = re.compile(r"[A-Za-z]{3,}")
LATIN_LETTER_RE = re.compile(r"[A-Za-z]")
EMOJI_RE = re.compile(
    "["
    "\U0001F1E6-\U0001F1FF"
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F7FF"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\U00002600-\U000026FF"
    "\U00002700-\U000027BF"
    "]+"
)


def estimate_tokens(text):
    jp = len(JP_RE.findall(text))
    emoji = len(EMOJI_RE.findall(text))
    words = LATIN_WORD_RE.findall(text)
    return jp + 1.3 * len(words) + 2 * emoji


pieces = []
for variant, (ja_column, en_column) in VARIANTS.items():
    for language, column, base_column in (("ja", ja_column, "text"), ("en", en_column, "text_en")):
        # Keep `base` as its own series: for the original prompts the prompt column and the base column are the same.
        piece = prompts.loc[prompts[column].notna(), ["source_id", "source_split", "risk_area_en", "harm_type_en", column]].copy()
        piece = piece.rename(columns={column: "prompt"})
        piece["base"] = prompts.loc[piece.index, base_column]
        piece["variant"] = variant
        piece["language"] = language
        pieces.append(piece)
long = pd.concat(pieces, ignore_index=True)
long["variant"] = pd.Categorical(long["variant"], list(VARIANTS), ordered=True)

long["chars"] = long["prompt"].str.len()
long["base_chars"] = long["base"].str.len()
long["jp_chars"] = long["prompt"].map(lambda text: len(JP_RE.findall(text)))
long["kana"] = long["prompt"].map(lambda text: len(KANA_RE.findall(text)))
long["latin_letters"] = long["prompt"].map(lambda text: len(LATIN_LETTER_RE.findall(text)))
long["latin_words"] = long["prompt"].map(lambda text: len(LATIN_WORD_RE.findall(text)))
long["emoji"] = long["prompt"].map(lambda text: len(EMOJI_RE.findall(text)))
long["tokens"] = long["prompt"].map(estimate_tokens)
long["token_source"] = "estimate"

long["flake"] = long["variant"].astype(str).map(FLAKE_TEXT)
added = long["chars"] - long["base_chars"]
long["insertions"] = np.where(long["flake"].notna(), (added / long["flake"].str.len()).round(), np.nan)
long["expected_insertions"] = np.where(
    long["variant"].astype(str).str.startswith("every10"), long["base_chars"] // 10,
    np.where(long["variant"].astype(str).str.startswith("quartile"), 4, np.nan),
)
# A yes-flake inside a Japanese prompt, or a はい-flake inside an English prompt, changes the script.
long["foreign_flake"] = ((long["language"] == "ja") & (long["flake"] == "yes")) | ((long["language"] == "en") & (long["flake"] == "はい"))
print(f"{len(long)} prompts across {long['variant'].nunique()} variants")

In [ ]:
# Exact Tiny Aya counts, including the chat template the eval sends. Skipped when the tokenizer is not on this machine.
tokenizer = None
try:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("CohereLabs/tiny-aya-water")
except Exception as error:
    print(f"Keeping the character estimate ({type(error).__name__}: {error}).")


def aya_tokens(text):
    try:
        rendered = tokenizer.apply_chat_template(
            [{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)
        return len(tokenizer(rendered, add_special_tokens=False)["input_ids"])
    except Exception:
        return len(tokenizer.encode(text))


if tokenizer is not None:
    long["tokens"] = long["prompt"].map(aya_tokens)
    long["token_source"] = "tiny-aya"
    print(f"Counted {len(long)} prompts with {tokenizer.name_or_path}")
long["token_source"].value_counts()

In [ ]:
long["over_context"] = long["tokens"] > CONTEXT
long["squeezes_answer"] = long["tokens"] + ANSWER_CAP > CONTEXT
long["outside_window"] = long["tokens"] > SLIDING_WINDOW
long["variant"] = long["variant"].astype(str)

# Aya answers that hit the 1,024-token cap were stored and then left unscored.
if EVAL_NAME in local_paths:
    connection = sqlite3.connect(local_paths[EVAL_NAME])
    answers = pd.read_sql_query(
        "SELECT source_id, source_split, variant, language, truncated FROM answers", connection)
    connection.close()
    answers["aya_truncated"] = answers["truncated"].map({1: True, 0: False, "1": True, "0": False, True: True, False: False})
    long = long.merge(
        answers[["source_id", "source_split", "variant", "language", "aya_truncated"]],
        on=["source_id", "source_split", "variant", "language"], how="left")
    print(f"Joined Aya truncation from {EVAL_NAME}: {long['aya_truncated'].notna().sum()} of {len(long)} prompts")
else:
    long["aya_truncated"] = np.nan
    print(f"No {EVAL_NAME}; Aya answer-cutoff rates are skipped.")

# Qwen rewrites that hit their own token cap are incomplete prompts.
rewrite_frames = []
for variant, name in QWEN_DBS.items():
    if name not in local_paths:
        continue
    connection = sqlite3.connect(local_paths[name])
    columns = [row[1] for row in connection.execute("PRAGMA table_info(records)")]
    ja_flag = [column for column in columns if column.endswith("_ja_truncated")]
    en_flag = [column for column in columns if column.endswith("_en_truncated")]
    if len(ja_flag) != 1 or len(en_flag) != 1:
        connection.close()
        print(f"{name}: no single _ja_truncated/_en_truncated pair ({ja_flag}, {en_flag})")
        continue
    flags = pd.read_sql_query(
        f'SELECT source_id, source_split, "{ja_flag[0]}" AS ja, "{en_flag[0]}" AS en FROM records', connection)
    connection.close()
    flags = flags.melt(id_vars=["source_id", "source_split"], value_vars=["ja", "en"],
                       var_name="language", value_name="rewrite_truncated")
    flags["variant"] = variant
    flags["rewrite_truncated"] = flags["rewrite_truncated"].map(
        {1: True, 0: False, "1": True, "0": False, True: True, False: False})
    rewrite_frames.append(flags)
    print(f"{name}: rewrite truncated on {int((flags['rewrite_truncated'] == True).sum())} of {len(flags)} sides")  # noqa: E712

if rewrite_frames:
    rewrites = pd.concat(rewrite_frames, ignore_index=True)
    long = long.merge(rewrites, on=["source_id", "source_split", "variant", "language"], how="left")
else:
    long["rewrite_truncated"] = np.nan

long["variant"] = pd.Categorical(long["variant"], list(VARIANTS), ordered=True)


## 4. Length and cutoff

`outside_window`: over 4,096 tokens, so the start of the prompt falls outside the sliding window of most layers. `squeezes_answer`: the prompt plus a 1,024-token answer does not fit in 8,192. `over_context`: the prompt alone does not fit. `aya_truncated`: the answer itself hit the 1,024-token cap and was dropped from scoring.

In [ ]:
length = long.groupby(["variant", "language"], observed=True).agg(
    prompts=("tokens", "size"),
    median_chars=("chars", "median"),
    median_tokens=("tokens", "median"),
    p95_tokens=("tokens", lambda column: column.quantile(0.95)),
    max_tokens=("tokens", "max"),
    outside_window=("outside_window", "mean"),
    squeezes_answer=("squeezes_answer", "mean"),
    over_context=("over_context", "mean"),
    aya_truncated=("aya_truncated", "mean"),
    rewrite_truncated=("rewrite_truncated", "mean"),
).round(3)
length

In [ ]:
token_plot = long.groupby(["variant", "language"], observed=True)["tokens"].median().unstack("language").reindex(list(VARIANTS))
axes = token_plot.plot.bar(figsize=(9, 3.5), rot=30, color=["#c45c26", "#3d6b99"])
axes.axhline(SLIDING_WINDOW, color="#666666", linestyle="--", linewidth=1)
axes.set_ylabel("median tokens")
axes.set_title("Median prompt length")
axes.legend(title="prompt language")
plt.tight_layout()
plt.show()

In [ ]:
# Longest prompts. These are the ones Aya is most likely to cut.
columns = ["source_id", "variant", "language", "risk_area_en", "tokens", "chars", "outside_window", "squeezes_answer", "aya_truncated", "prompt"]
long.nlargest(8, "tokens")[columns]

## 5. Emoji

A pair is one-sided when one language has an emoji and the other does not.

In [ ]:
long.groupby(["variant", "language"], observed=True).agg(
    prompts=("emoji", "size"),
    with_emoji=("emoji", lambda column: int((column > 0).sum())),
    emoji_rate=("emoji", lambda column: (column > 0).mean()),
    median_emoji=("emoji", "median"),
).round(3)

In [ ]:
emoji_rows = long[long["emoji"] > 0][["source_id", "source_split", "variant", "language", "emoji", "risk_area_en", "prompt"]]
print(f"{len(emoji_rows)} prompts contain an emoji")
emoji_rows.head(8)

## 6. Is the pair a Japanese vs English contrast?

For each row and variant, both sides have to be the same request in two languages. The pair fails that when:

- the Japanese side has 5 or more Latin words, or the English side has any kana (`script`)
- an emoji is on one side only
- one side is over the 4,096-token window and the other is not, or either side does not leave room for the answer
- the token counts differ by more than 2×
- a flake variant inserts a different number of flakes, or inserts a flake in the other language's script
- Qwen cut off one rewrite and not the other, or Aya cut off one answer and not the other

`clean` is the share of pairs with none of those.

In [ ]:
values = ["tokens", "chars", "emoji", "latin_words", "kana", "insertions", "foreign_flake",
          "outside_window", "squeezes_answer", "over_context", "aya_truncated", "rewrite_truncated", "prompt"]
wide = long.pivot_table(
    index=["source_id", "source_split", "variant", "risk_area_en", "harm_type_en"],
    columns="language", values=values, aggfunc="first", observed=True)
wide.columns = [f"{name}_{language}" for name, language in wide.columns]
pairs = wide.reset_index()
pairs["variant"] = pd.Categorical(pairs["variant"], list(VARIANTS), ordered=True)
# pivot_table drops a value column when every entry is missing, which is the case before the eval or Qwen dbs are joined.
for name in ("aya_truncated", "rewrite_truncated"):
    for language in ("ja", "en"):
        column = f"{name}_{language}"
        if column not in pairs:
            pairs[column] = np.nan

both = pairs["prompt_ja"].notna() & pairs["prompt_en"].notna()
pairs = pairs[both].copy()
pairs["script_ja"] = pairs["latin_words_ja"] >= 5
pairs["script_en"] = pairs["kana_en"] >= 1
pairs["script"] = pairs["script_ja"] | pairs["script_en"]
pairs["emoji_one_sided"] = (pairs["emoji_ja"] > 0) != (pairs["emoji_en"] > 0)
pairs["window_one_sided"] = pairs["outside_window_ja"].fillna(False) != pairs["outside_window_en"].fillna(False)
pairs["squeezes_either"] = pairs["squeezes_answer_ja"].fillna(False) | pairs["squeezes_answer_en"].fillna(False)
pairs["over_context_either"] = pairs["over_context_ja"].fillna(False) | pairs["over_context_en"].fillna(False)
ratio = pairs["tokens_ja"] / pairs["tokens_en"].replace(0, np.nan)
pairs["token_ratio"] = ratio
pairs["unbalanced_length"] = (ratio < 0.5) | (ratio > 2)
insertion_ratio = pairs["insertions_ja"] / pairs["insertions_en"].replace(0, np.nan)
pairs["unbalanced_flakes"] = pairs["insertions_ja"].notna() & ((insertion_ratio < 0.5) | (insertion_ratio > 2) | (pairs["insertions_en"] == 0))
pairs["foreign_flake_either"] = pairs["foreign_flake_ja"].fillna(False) | pairs["foreign_flake_en"].fillna(False)
pairs["rewrite_one_sided"] = pairs["rewrite_truncated_ja"].notna() & pairs["rewrite_truncated_en"].notna() & (
    pairs["rewrite_truncated_ja"] != pairs["rewrite_truncated_en"])
pairs["aya_one_sided"] = pairs["aya_truncated_ja"].notna() & pairs["aya_truncated_en"].notna() & (
    pairs["aya_truncated_ja"] != pairs["aya_truncated_en"])

issue_columns = ["script", "emoji_one_sided", "window_one_sided", "squeezes_either", "over_context_either",
                 "unbalanced_length", "unbalanced_flakes", "foreign_flake_either", "rewrite_one_sided", "aya_one_sided"]
pairs["clean"] = ~pairs[issue_columns].any(axis=1)
print(f"{len(pairs)} pairs with both languages")

In [ ]:
issues = pairs.groupby("variant", observed=True)[issue_columns + ["clean"]].mean().reindex(list(VARIANTS)).round(3)
issues.insert(0, "pairs", pairs.groupby("variant", observed=True).size().reindex(list(VARIANTS)))
issues

In [ ]:
# Flake counts follow character length, so the longer language receives more insertions.
flake_pairs = pairs[pairs["variant"].isin(FLAKE_TEXT)]
flake_pairs.groupby("variant", observed=True)[["insertions_ja", "insertions_en", "chars_ja", "chars_en"]].median().round(1)

## 7. Within each category, and across categories

A risk area whose Japanese and English prompts differ in length, script, or emoji is not a clean language comparison. A gap between risk areas means a category effect can be that structural difference.

In [ ]:
def category_table(frame, category):
    grouped = frame.groupby(category, observed=True)
    table = grouped.agg(
        pairs=("clean", "size"),
        clean=("clean", "mean"),
        script=("script", "mean"),
        emoji_one_sided=("emoji_one_sided", "mean"),
        unbalanced_length=("unbalanced_length", "mean"),
        window_one_sided=("window_one_sided", "mean"),
        squeezes_either=("squeezes_either", "mean"),
        aya_one_sided=("aya_one_sided", "mean"),
        rewrite_one_sided=("rewrite_one_sided", "mean"),
        median_token_ratio=("token_ratio", "median"),
    )
    return table.sort_values("clean").round(3)


original_pairs = pairs[pairs["variant"] == "original"]
print("Original prompts by risk area")
display(category_table(original_pairs, "risk_area_en"))
print("Original prompts by harm type, areas with at least 20 pairs")
harm = category_table(original_pairs, "harm_type_en")
harm[harm["pairs"] >= 20]

In [ ]:
# How far the risk areas sit from each other, on the original prompts and on every variant.
def spread(frame):
    by_area = frame.groupby("risk_area_en", observed=True)[["clean", "script", "emoji_one_sided", "unbalanced_length", "token_ratio"]].mean()
    return pd.DataFrame({
        "lowest_area": by_area.min(),
        "median_area": by_area.median(),
        "highest_area": by_area.max(),
        "highest_minus_lowest": by_area.max() - by_area.min(),
    }).round(3)


spread_by_variant = {variant: spread(group) for variant, group in pairs.groupby("variant", observed=True)}
print("Spread of risk-area rates on the original prompts")
display(spread_by_variant["original"])
pd.concat({variant: table.loc["clean"] for variant, table in spread_by_variant.items()}, axis=1).T.round(3)

In [ ]:
# Clean-pair rate by risk area (rows) and variant (columns).
clean_by_area = pairs.pivot_table(index="risk_area_en", columns="variant", values="clean", aggfunc="mean", observed=True)
clean_by_area = clean_by_area.reindex(columns=list(VARIANTS))
clean_by_area.insert(0, "pairs", original_pairs.groupby("risk_area_en").size())
clean_by_area.sort_values("original").round(2)

## 8. Examples

Change `VARIANT` to inspect another prompt variant. Each frame is a pair that fails one check.

In [ ]:
VARIANT = "original"
sample = pairs[pairs["variant"] == VARIANT]
show = ["source_id", "risk_area_en", "harm_type_en", "tokens_ja", "tokens_en", "emoji_ja", "emoji_en",
        "latin_words_ja", "kana_en", "prompt_ja", "prompt_en"]

print("Script leak")
display(sample[sample["script"]].head(5)[show])
print("Emoji on one side")
display(sample[sample["emoji_one_sided"]].head(5)[show])
print("Length ratio outside 0.5–2")
display(sample[sample["unbalanced_length"]].sort_values("token_ratio").head(5)[show])

## 9. Biases in the original dataset

These checks read only `answercarefully.sqlite3`. They describe what AnswerCarefully itself brings to a Japanese vs English comparison, before any model runs.

| Feature | Column | Why it matters |
|---|---|---|
| Category balance | `risk_area_en`, `source_split` | Pooled rates are dominated by the largest risk area (Malicious Uses). |
| Cultural adaptation | `adaptation_tag`, `translation_note`, `culture` | AnswerCarefully's English release tags items that were adapted for a Japanese context. In this snapshot, tag `0` items almost never carry a translator's note, tag `2` items almost always do, and tag `1` items are often about Japanese people, places, or institutions. For those items the English prompt is not the same request to an English speaker. |
| Japan mentioned | `mentions_japan` | Japan, Japanese, or a Japanese city named in either language. |
| Prompt form | `prompt_form`, `first_person` | Question, request, or statement; whether the user talks about themselves. Heuristic, read from the Japanese text. |
| Length | `chars_ja`, `chars_en`, `en_ja_char_ratio`, `length_band` | Length quartile of the Japanese prompt. |
| Reference answer | `reference_declines`, `reference_chars` | Whether the Japanese reference answer declines. Heuristic: it contains a phrase such as できません or お答えできません. A reference that answers with care means a refusal is not the expected behaviour. |

The last cell saves these features to `answercarefully_bias_features.sqlite3` in the same folder, so `preview_eval.ipynb` can split Aya's results by them.

In [ ]:
import json

from scipy.stats import chi2_contingency

connection = sqlite3.connect(local_paths[ORIGINAL_NAME])
original = pd.read_sql_query(
    "SELECT source_id, source_split, text, text_en, output, risk_area_en, harm_type_en, specific_harm_en, meta_mlmc FROM records",
    connection)
connection.close()

JAPAN_EN = re.compile(r"\b(?:Japan|Japanese|Tokyo|Osaka|Kyoto)\b", re.IGNORECASE)
JAPAN_JA = re.compile(r"日本|東京|大阪|京都")
QUESTION = re.compile(r"[?？]|(か|かな|かしら|の)[\s。」』）)]*$")
REQUEST_END = re.compile(r"(ください|下さい|教えて|て|くれ|しろ|せよ|なさい|てほしい|て欲しい|方法|やり方|手順|書き方|作り方|翻訳)[。！!\s」』）)ｗw]*$")
FIRST_PERSON = re.compile(r"私|僕|俺|わたし|あたし")
# Japanese reference answers that decline. The rest answer, warn, or redirect.
DECLINE = re.compile(r"できません|できかねます|お答えでき|お応えでき|お手伝いでき|控えさせて|差し控え")


def prompt_form(text):
    text = text.strip()
    if QUESTION.search(text):
        return "question"
    if REQUEST_END.search(text):
        return "request"
    return "statement"


mlmc = original["meta_mlmc"].map(json.loads)
features = original[["source_id", "source_split", "risk_area_en", "harm_type_en", "specific_harm_en"]].copy()
features["adaptation_tag"] = mlmc.map(lambda meta: str(meta.get("adaptation-tag", "")).strip() or "unknown")
features["translation_note"] = mlmc.map(lambda meta: bool(str(meta.get("translation-notes") or "").strip()))
features["culture"] = np.where(features["adaptation_tag"].isin(["1", "2"]), "adapted for Japan", "plain translation")
features["mentions_japan"] = original["text"].str.contains(JAPAN_JA) | original["text_en"].fillna("").str.contains(JAPAN_EN)
features["prompt_form"] = original["text"].map(prompt_form)
features["first_person"] = original["text"].str.contains(FIRST_PERSON)
features["chars_ja"] = original["text"].str.len()
features["chars_en"] = original["text_en"].str.len()
features["en_ja_char_ratio"] = (features["chars_en"] / features["chars_ja"]).round(3)
features["length_band"] = "Q" + (pd.qcut(features["chars_ja"], 4, labels=False, duplicates="drop") + 1).astype(str)
features["reference_declines"] = original["output"].str.contains(DECLINE)
features["reference_chars"] = original["output"].str.len()
features["duplicate_text"] = original["text"].duplicated(keep=False)

print(f"{len(features)} items, {features['duplicate_text'].sum()} share their Japanese text with another item")
summary_columns = ["adaptation_tag", "culture", "translation_note", "mentions_japan", "prompt_form", "first_person", "length_band", "reference_declines"]
pd.concat({column: features[column].astype(str).value_counts(normalize=True) for column in summary_columns}).round(3).rename("share").to_frame()

### 9.1 Category balance

Item counts per risk area in each split, and the share of the split. A chi-square test checks whether dev and test have the same mix. Specific harms with fewer than 30 items give noisy per-harm rates.

In [ ]:
balance = pd.crosstab(features["risk_area_en"], features["source_split"], margins=True, margins_name="all")
shares = (balance / balance.loc["all"] * 100).round(1).add_suffix(" %")
display(balance.join(shares))
chi2, p, dof, _ = chi2_contingency(pd.crosstab(features["risk_area_en"], features["source_split"]))
print(f"Risk-area mix, dev vs test: chi2 = {chi2:.1f}, dof = {dof}, p = {p:.3g}")

specific = features["specific_harm_en"].value_counts()
print(f"{(specific < 30).sum()} of {len(specific)} specific harms have fewer than 30 items "
      f"(smallest: {specific.min()}, largest: {specific.max()}).")
features["harm_type_en"].value_counts().rename("items").to_frame()

### 9.2 Cultural adaptation

Share of each risk area by adaptation tag. A risk area that is mostly adapted for Japan compares a Japanese request with an English request that has lost its context, so a language gap there can be a cultural gap.

In [ ]:
culture_by_area = pd.crosstab(features["risk_area_en"], features["adaptation_tag"], normalize="index").round(3)
culture_by_area.insert(0, "items", features.groupby("risk_area_en").size())
culture_by_area["translation_note"] = features.groupby("risk_area_en")["translation_note"].mean().round(3)
culture_by_area["mentions_japan"] = features.groupby("risk_area_en")["mentions_japan"].mean().round(3)
display(culture_by_area)

# Specific harms that are mostly adapted for Japan.
adapted = features.groupby("specific_harm_en").agg(items=("culture", "size"),
                                                   adapted=("culture", lambda column: (column == "adapted for Japan").mean()))
adapted[adapted["items"] >= 15].sort_values("adapted", ascending=False).head(10).round(2)

In [ ]:
# Adapted items with their translator's note, one per risk area.
notes = original.assign(note=mlmc.map(lambda meta: str(meta.get("translation-notes") or "").strip()))
notes = notes.join(features[["adaptation_tag"]])
sample = notes[(notes["adaptation_tag"] == "2") & (notes["note"] != "")].groupby("risk_area_en").head(1)
sample[["risk_area_en", "text", "text_en", "note"]]

### 9.3 Prompt form and length

How each risk area asks. A risk area made of direct requests and another made of statements or personal stories test different things, whatever the language.

In [ ]:
form_by_area = pd.crosstab(features["risk_area_en"], features["prompt_form"], normalize="index").round(3)
form_by_area["first_person"] = features.groupby("risk_area_en")["first_person"].mean().round(3)
form_by_area["median_chars_ja"] = features.groupby("risk_area_en")["chars_ja"].median()
form_by_area["median_chars_en"] = features.groupby("risk_area_en")["chars_en"].median()
form_by_area["median_en_ja_ratio"] = features.groupby("risk_area_en")["en_ja_char_ratio"].median().round(2)
display(form_by_area)

# English length per Japanese character: adapted items often grow in translation.
features.groupby("adaptation_tag")["en_ja_char_ratio"].describe(percentiles=[0.1, 0.5, 0.9]).round(2)

### 9.4 What the reference answer expects

AnswerCarefully's references are model answers written by people. Where most references decline, refusal is the expected behaviour. Where most answer with care, a refusal is over-refusal. Refusal rate alone cannot tell these apart.

In [ ]:
reference = features.pivot_table(index="risk_area_en", columns="culture", values="reference_declines", aggfunc="mean").round(3)
reference.insert(0, "declines_overall", features.groupby("risk_area_en")["reference_declines"].mean().round(3))
reference["median_reference_chars"] = features.groupby("risk_area_en")["reference_chars"].median()
display(reference)
features.groupby("prompt_form")["reference_declines"].agg(["size", "mean"]).round(3)

### 9.5 Which features move with the category

Cramér's V between each feature and the risk area (0 = independent, 1 = one determines the other), and between each feature and the split. A feature with a high V against risk area is entangled with it: a difference between risk areas may be that feature instead.

In [ ]:
def association(a, b):
    table = pd.crosstab(a, b)
    if min(table.shape) < 2:
        return np.nan, np.nan
    chi2, p, _, _ = chi2_contingency(table)
    return np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1))), p


BIAS_FEATURES = ["adaptation_tag", "translation_note", "mentions_japan", "prompt_form", "first_person", "length_band", "reference_declines"]
rows = []
for feature in BIAS_FEATURES:
    v_area, p_area = association(features[feature], features["risk_area_en"])
    v_split, p_split = association(features[feature], features["source_split"])
    rows.append({"feature": feature, "cramers_v_risk_area": v_area, "p_risk_area": p_area,
                 "cramers_v_split": v_split, "p_split": p_split})
pd.DataFrame(rows).set_index("feature").sort_values("cramers_v_risk_area", ascending=False).round(3)

### 9.6 Save the features for the eval preview

In [ ]:
from datetime import datetime, timezone

FEATURES_NAME = "answercarefully_bias_features.sqlite3"
local_features = WORK_DIR / FEATURES_NAME
local_features.unlink(missing_ok=True)
connection = sqlite3.connect(local_features)
with connection:
    features.to_sql("features", connection, index=False)
    pd.DataFrame([{"source": ORIGINAL_NAME, "features": ", ".join(features.columns),
                   "created_at": datetime.now(timezone.utc).isoformat()}]).to_sql("features_info", connection, index=False)
connection.close()
shutil.copy(local_features, DRIVE_DIR / FEATURES_NAME)
print(f"Saved {DRIVE_DIR / FEATURES_NAME}")